# Credit Risk EDA

Dataset: Credit Risk Benchmark Dataset
Target: `dlq_2yrs` (1 = delinquent in 2 years)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = '../data/raw/Credit Risk Benchmark Dataset.csv'
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
df.head()

## Data Quality Check

In [ ]:
df.info()

In [ ]:
missing = df.isnull().sum().rename('missing')
pct_missing = (missing / len(df) * 100).rename('pct_missing')
pd.concat([missing, pct_missing, df.dtypes.rename('dtype')], axis=1)

## Target Distribution

In [ ]:
target_counts = df['dlq_2yrs'].value_counts().sort_index()
target_counts

In [ ]:
labels = ['Not Delinquent (0)', 'Delinquent (1)']
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(labels, target_counts.values, color=['steelblue', 'tomato'])
ax.bar_label(bars)
ax.set_title('Target Class Distribution')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## Numeric Summary

In [ ]:
df.describe().T

## Feature Histograms

In [ ]:
features = [c for c in df.columns if c != 'dlq_2yrs']
n_cols = 3
n_rows = -(-len(features) // n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(features):
    axes[i].hist(df[col], bins=40, color='steelblue', alpha=0.8)
    axes[i].set_title(col)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

## Log Transform Right-Skewed Variables

We apply `log1p` to non-negative features with skewness greater than 1.0.

In [ ]:
numeric_features = [c for c in df.columns if c != 'dlq_2yrs']
skewness = df[numeric_features].skew().sort_values(ascending=False)

# Only transform non-negative features to keep log1p safe and interpretable.
right_skewed = [
    c for c in skewness.index
    if skewness[c] > 1.0 and df[c].min() >= 0
]

df_log = df.copy()
for col in right_skewed:
    df_log[col] = np.log1p(df_log[col])

pd.DataFrame({
    'skew_before': df[right_skewed].skew(),
    'skew_after_log1p': df_log[right_skewed].skew(),
}).sort_values('skew_before', ascending=False)

In [ ]:
if right_skewed:
    n_cols = 3
    n_rows = -(-len(right_skewed) // n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
    axes = np.array(axes).reshape(-1)

    for i, col in enumerate(right_skewed):
        ax = axes[i]
        ax.hist(df[col], bins=40, alpha=0.45, label='before', color='tomato')
        ax.hist(df_log[col], bins=40, alpha=0.45, label='log1p', color='steelblue')
        ax.set_title(col)
        ax.legend(fontsize=8)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.show()
else:
    print('No right-skewed non-negative variables found with threshold > 1.0')

## Correlation Matrix (After Log Transform)

In [ ]:
corr = df_log.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
fig.colorbar(im, ax=ax)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.columns)
ax.set_title('Correlation Matrix (log1p transformed where applicable)')
plt.tight_layout()
plt.show()

## Quick Notes
- Right-skewed non-negative variables were transformed with `log1p`.
- Use `df_log` for modeling to reduce outlier dominance and improve linear model stability.
- Next: train/validation split, baseline Logistic Regression and tree models.